# IBM AML EDA (Kaggle)

Dataset: [IBM Transactions for Anti-Money Laundering](https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml).

- Place unpacked transaction CSVs under `data/raw/`.
- Optional: add `data/raw/patterns.txt` (one pattern per line; `#` starts a comment).
- **File selection** below supports a single CSV, a list of basenames, or all `*.csv` in `data/raw/`.
- Canonical columns after modeling live in `src/aml_inspector/data/datasets.py`; raw CSV expectations are in `src/aml_inspector/data/column_contract.py`.

In [ ]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import pandas as pd

# Notebook-friendly import when package is not installed
_here = Path.cwd().resolve()
_root = _here
for _ in range(6):
    if (_root / "pyproject.toml").is_file():
        break
    _root = _root.parent
else:
    _root = _here
_src = _root / "src"
if _src.is_dir() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from aml_inspector.config import DATA_RAW
from aml_inspector.data.column_contract import (
    COL_AMOUNT_PAID,
    COL_AMOUNT_RECEIVED,
    COL_FROM_BANK,
    COL_IS_LAUNDERING,
    COL_PAYMENT_FORMAT,
    COL_PAYMENT_TYPE,
    COL_PAYMENT_CURRENCY,
    COL_RECEIVING_CURRENCY,
    COL_SOURCE_COUNTRY,
    COL_TARGET_COUNTRY,
    COL_TIMESTAMP,
    COL_TO_BANK,
    normalize_account_column_names,
    validate_raw_csv,
)

DATA_RAW = DATA_RAW.resolve()
PATTERNS_FILE = DATA_RAW / "patterns.txt"
print("DATA_RAW:", DATA_RAW)

## Discover files in `data/raw`

Lists `*.csv` and whether `patterns.txt` exists.

In [ ]:
csv_paths = sorted(DATA_RAW.glob("*.csv"))
print(f"Found {len(csv_paths)} CSV file(s) under data/raw")
for p in csv_paths:
    print(" -", p.name)
print("patterns.txt:", "yes" if PATTERNS_FILE.is_file() else "no (optional)", "->", PATTERNS_FILE)

## Select CSV(s) for EDA

- Set `ALL_CSVS = True` to profile every `*.csv` in `data/raw/`.
- Or set `ALL_CSVS = False` and set `SELECTED_FILES` to a **single** basename (`"HI-Small_Trans.csv"`) or a **list** of basenames.
- `SAMPLE_NROWS`: rows read per file for in-memory plots/tables (use `None` for full read — can be slow/large).
- `CHUNK_SIZE`: used for optional full-scan summaries without loading entire files into RAM.

In [ ]:
ALL_CSVS = True
SELECTED_FILES: str | list[str] = [
    "HI-Small_Trans.csv",
    "HI-Medium_Trans.csv",
]

SAMPLE_NROWS = 50_000
CHUNK_SIZE = 200_000

if ALL_CSVS:
    selected_paths = sorted(DATA_RAW.glob("*.csv"))
else:
    if isinstance(SELECTED_FILES, str):
        names = [SELECTED_FILES]
    else:
        names = list(SELECTED_FILES)
    selected_paths = []
    for name in names:
        p = Path(name)
        selected_paths.append(p if p.is_file() else (DATA_RAW / p.name).resolve())

missing = [str(p) for p in selected_paths if not p.is_file()]
if missing:
    raise FileNotFoundError("CSV(s) not found: " + ", ".join(missing))
if not selected_paths:
    raise FileNotFoundError(f"No CSVs selected. Place files in {DATA_RAW}")

for p in selected_paths:
    validate_raw_csv(p)
print("Selected:")
for p in selected_paths:
    print(" -", p)

## EDA helpers (reusable patterns)

Small utilities: label coercion, schema profile, chunked aggregates, pattern matching on text columns.

In [ ]:
def normalize_label(series: pd.Series) -> pd.Series:
    if series.dtype == object:
        lower = series.astype(str).str.strip().str.lower()
        return lower.isin(("1", "true", "yes", "y"))
    return series.astype("Int64").fillna(0).astype(int).astype(bool)


def add_engineering_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    rename = normalize_account_column_names(out.columns)
    out = out.rename(columns=rename)
    if COL_IS_LAUNDERING in out.columns:
        out["_label_bool"] = normalize_label(out[COL_IS_LAUNDERING])
    if COL_TIMESTAMP in out.columns:
        out["_event_ts"] = pd.to_datetime(out[COL_TIMESTAMP], errors="coerce", utc=True)
    return out


def profile_schema(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for c in df.columns:
        s = df[c]
        rows.append(
            {
                "column": c,
                "dtype": str(s.dtype),
                "non_null": int(s.notna().sum()),
                "null_pct": round(100.0 * s.isna().mean(), 4),
                "n_unique": int(s.nunique(dropna=True)),
            }
        )
    return pd.DataFrame(rows).sort_values("null_pct", ascending=False)


def read_sample(path: Path, *, nrows: int | None) -> pd.DataFrame:
    kwargs: dict = {"low_memory": False}
    if nrows is not None:
        kwargs["nrows"] = nrows
    df = pd.read_csv(path, **kwargs)
    df["_source_file"] = path.name
    return df


def chunked_file_summary(path: Path, *, chunksize: int) -> dict:
    """Row count, label counts, min/max timestamp without full load."""
    total = 0
    pos = neg = 0
    ts_min: pd.Timestamp | None = None
    ts_max: pd.Timestamp | None = None
    for chunk in pd.read_csv(path, chunksize=chunksize, low_memory=False):
        total += len(chunk)
        if COL_IS_LAUNDERING in chunk.columns:
            y = normalize_label(chunk[COL_IS_LAUNDERING])
            pos += int(y.sum())
            neg += int((~y).sum())
        if COL_TIMESTAMP in chunk.columns:
            ts = pd.to_datetime(chunk[COL_TIMESTAMP], errors="coerce", utc=True)
            cmin, cmax = ts.min(), ts.max()
            if pd.notna(cmin):
                ts_min = cmin if ts_min is None else min(ts_min, cmin)
            if pd.notna(cmax):
                ts_max = cmax if ts_max is None else max(ts_max, cmax)
    out = {
        "rows": total,
        "laundering_rows": pos,
        "non_laundering_rows": neg,
        "laundering_rate": (pos / total) if total else None,
    }
    if ts_min is not None and ts_max is not None:
        out["timestamp_min"] = ts_min
        out["timestamp_max"] = ts_max
    return out


def text_columns(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if df[c].dtype == object or str(df[c].dtype) == "string"]


def pattern_hits(df: pd.DataFrame, pattern: str, *, case: bool = False) -> pd.Series:
    """Any-row boolean mask if pattern appears in any object column (regex)."""
    if not pattern.strip():
        return pd.Series(False, index=df.index)
    masks = []
    for c in text_columns(df):
        s = df[c].astype(str)
        masks.append(s.str.contains(pattern, case=case, na=False, regex=True))
    if not masks:
        return pd.Series(False, index=df.index)
    m = masks[0]
    for x in masks[1:]:
        m = m | x
    return m


def load_patterns_file(path: Path) -> list[str]:
    if not path.is_file():
        return []
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    out: list[str] = []
    for line in lines:
        s = line.split("#", 1)[0].strip()
        if s:
            out.append(s)
    return out

## Load samples (per selected file)

Combined frame includes `_source_file` for faceted comparisons.

In [ ]:
samples = [read_sample(p, nrows=SAMPLE_NROWS) for p in selected_paths]
raw_df = pd.concat(samples, ignore_index=True)
df = add_engineering_columns(raw_df)
print("Combined sample shape:", df.shape)
df.head(3)

## Schema, missingness, duplicates

In [ ]:
display(profile_schema(df).head(25))
dup_all = int(df.duplicated().sum())
cols_no_meta = [c for c in df.columns if not str(c).startswith("_")]
dup_business = int(df[cols_no_meta].duplicated().sum()) if cols_no_meta else 0
print("Duplicate rows (all columns):", dup_all)
print("Duplicate rows (non-meta columns):", dup_business)

## Target balance (`Is Laundering`)

In [ ]:
if "_label_bool" in df.columns:
    vc = df["_label_bool"].value_counts(dropna=False)
    display(vc.to_frame("count"))
    display((vc / vc.sum()).rename("fraction").to_frame())
else:
    print("No laundering column in sample.")

## Timestamps (range and daily volume in sample)

In [ ]:
if "_event_ts" in df.columns:
    ts = df["_event_ts"]
    print("min:", ts.min(), "max:", ts.max(), "nulls:", int(ts.isna().sum()))
    daily = df.assign(_d=ts.dt.floor("D")).groupby("_d", dropna=True).size()
    display(daily.tail(10).to_frame("rows_in_sample"))
else:
    print("No timestamp column in sample.")

## Categoricals: payment type/format, currencies, countries

In [ ]:
def top_values(col: str, n: int = 15) -> pd.DataFrame | None:
    if col not in df.columns:
        return None
    return (
        df[col]
        .astype(str)
        .value_counts(dropna=False)
        .head(n)
        .rename("count")
        .to_frame()
    )


for col in (
    COL_PAYMENT_TYPE,
    COL_PAYMENT_FORMAT,
    COL_PAYMENT_CURRENCY,
    COL_RECEIVING_CURRENCY,
    COL_SOURCE_COUNTRY,
    COL_TARGET_COUNTRY,
):
    t = top_values(col)
    if t is not None:
        print("\n===", col, "===")
        display(t)

## Amounts (received / paid)

In [ ]:
for col in (COL_AMOUNT_RECEIVED, COL_AMOUNT_PAID):
    if col not in df.columns:
        continue
    x = pd.to_numeric(df[col], errors="coerce")
    print(col, "describe:")
    display(x.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_frame())

## Banks and accounts (cardinality and top banks)

In [ ]:
for col in (COL_FROM_BANK, COL_TO_BANK):
    if col not in df.columns:
        continue
    s = pd.to_numeric(df[col], errors="coerce")
    print(col, "n_unique (non-null):", int(s.nunique(dropna=True)))
    display(s.value_counts(dropna=True).head(15).rename("rows").to_frame())

for ac in ("from_account_raw", "to_account_raw"):
    if ac in df.columns:
        print(ac, "n_unique:", df[ac].nunique(dropna=True))

## Multi-file comparison (chunked full scan)

Uses `CHUNK_SIZE` so each file is summarized without loading it entirely.

In [ ]:
rows = []
for p in selected_paths:
    s = chunked_file_summary(p, chunksize=CHUNK_SIZE)
    s["file"] = p.name
    rows.append(s)
cmp = pd.DataFrame(rows)
display(cmp)

## `patterns.txt` (optional)

Loads non-comment lines from `data/raw/patterns.txt` and reports how many **sample** rows match each pattern (regex) in any text column.

In [ ]:
patterns = load_patterns_file(PATTERNS_FILE)
if not patterns:
    print(
        "No patterns loaded. Add lines to",
        PATTERNS_FILE,
        "(one pattern per line; # comments allowed).",
    )
else:
    print(f"Loaded {len(patterns)} pattern(s). Preview:")
    for i, pat in enumerate(patterns[:20], 1):
        print(f" {i:2d}.", pat[:120] + ("…" if len(pat) > 120 else ""))
    hit_rows = []
    for pat in patterns:
        try:
            m = pattern_hits(df, pat)
            hit_rows.append({"pattern": pat, "sample_hits": int(m.sum())})
        except re.error as e:
            hit_rows.append({"pattern": pat, "sample_hits": None, "regex_error": str(e)})
    display(pd.DataFrame(hit_rows))

## Per-file sample: laundering rate in the loaded sample

In [ ]:
if "_label_bool" in df.columns and "_source_file" in df.columns:
    g = df.groupby("_source_file", dropna=False)["_label_bool"]
    summary = pd.DataFrame(
        {
            "rows_in_sample": g.size(),
            "laundering_rate": g.mean(),
        }
    )
    display(summary)
else:
    print("Skipping: need _label_bool and _source_file.")